# Create solvated dialanine and test simulation

* **Input**: `data/ala2.pdb`
* **Output**: `ala2_solv.pdb` = (2.6 nm)$^3$ cell, centred, 538 water

Compare with mdshare: (2.32 nm)$^3$ cell, centred, 651 water

https://markovmodel.github.io/mdshare/ALA2/#alanine-dipeptide

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import mdtraj as md
import openmm as mm
import openmm.unit as unit
from openmm.app import Simulation, ForceField, PDBFile
from openmm.app.modeller import Modeller
from pathlib import Path

from src.param import SimulationParameters
from src.util import create_system, write_pdb
from src.util import state_data_reporter, hdf5_reporter
from src.util import print_status_file, print_status_files

working_dir = Path('data/setup')
p = SimulationParameters(
    working_dir = str(working_dir),
    simulation_time = 0.1 * unit.nanosecond,
)
pdb_file = working_dir / 'ala2.pdb'
pdb_center = working_dir / 'ala2_center_cell.pdb'
pdb_solv = working_dir / 'ala2_solv.pdb'
pdb_min = working_dir / 'ala2_solv_min.pdb'
pdb_equil = working_dir / 'ala2_solv_equil.pdb'
trajout = working_dir / 'traj.out'
trajh5 = working_dir / 'traj.h5'

pdb_files = [pdb_file, pdb_center, pdb_solv, pdb_min, pdb_equil, trajout, trajh5]
print_status_files(pdb_files)

file data/setup/ala2.pdb exsists.                 Modified Sat Aug  3 10:01:03 2024
file data/setup/ala2_center_cell.pdb exsists.     Modified Wed Jan  8 13:06:13 2025
file data/setup/ala2_solv.pdb exsists.            Modified Wed Jan  8 13:07:33 2025
file data/setup/ala2_solv_min.pdb exsists.        Modified Wed Jan  8 13:08:08 2025
file data/setup/ala2_solv_equil.pdb exsists.      Modified Wed Jan  8 13:12:42 2025
file data/setup/traj.out exsists.                 Modified Wed Jan  8 13:12:42 2025
file data/setup/traj.h5 exsists.                  Modified Wed Jan  8 13:12:42 2025


# 1. Centering

In [2]:
pdb = PDBFile(str(pdb_file))
traj = md.load_pdb(str(pdb_file))

# mdshare dataset = (2.3222 nm)^3
boxdim = (2.6, 2.6, 2.6)

# MDTraj stores positions in nm
xyz_nm = traj.xyz.squeeze() + np.array(boxdim) / 2

# PDB uses Angstroms
PDBFile.writeFile(pdb.topology, xyz_nm * 10, open(str(pdb_center), 'w'))

# 2. Solvation

In [3]:
pdb = PDBFile(str(pdb_center))
modeller = Modeller(pdb.topology, pdb.positions)
forcefield = ForceField('amber99sb.xml', 'tip3p.xml')
modeller.addSolvent(forcefield, boxSize=mm.Vec3(*boxdim) * unit.nanometers)

system = create_system(forcefield, modeller.topology)
integrator = mm.LangevinIntegrator(p.temperature, p.friction_coeff, p.timestep)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)
write_pdb(simulation, str(pdb_solv))

num_waters = 0
for r in modeller.topology.residues():
    if r.name == 'HOH':
        num_waters += 1
print(f'{num_waters = }, mdshare dataset = 651 water')

num_waters = 538, mdshare dataset = 651 water


# 3. Minimization

In [4]:
simulation.minimizeEnergy()
write_pdb(simulation, str(pdb_min))

# 4. Equilibration

In [9]:
sdr = state_data_reporter(str(trajout), p.report_interval)
hdr = hdf5_reporter(str(trajh5), p.report_interval)
simulation.reporters.append(sdr)
simulation.reporters.append(hdr)
simulation.step(10000)

write_pdb(simulation, str(pdb_equil))
for reporter in simulation.reporters:
    if hasattr(reporter, 'close'):
        reporter.close()

# 5. Copy final PDB file to usual place

In [20]:
pdb_final = Path(p.pdb_file)
print_status_file(pdb_final)

overwrite = False

if (not pdb_final.exists()) or (pdb_final.exists() and overwrite):
    print(f'writing to {str(pdb_final)}')
    # pdb_equil.copy(pdb_final)                  # requires python 3.14 (?)
    pdb_final.write_text(pdb_equil.read_text())  # old way

file data/ala2_solv.pdb exsists.     Modified Sat Aug  3 10:37:28 2024


In [ ]:
# import shutil   # older way

# shutil.copyfile(str(pdb_equil), p.pdb_file)